# 00 — Pipeline Audit & Raw Data Compliance

Notebook này kiểm tra dữ liệu đầu vào, format sample, target reconciliation, và ghi lại pipeline tổng thể. Không train model và không tạo submission.


In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def find_package_paths():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        # Final GitHub layout: repo_root/Raw Data and repo_root/part3_forecasting/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "part3_forecasting" / "notebooks").exists():
            return candidate, candidate / "part3_forecasting"
        # Backward-compatible layout used during development: package_root/Raw Data and package_root/notebooks
        if (candidate / "Raw Data").exists() and (candidate / "notebooks").exists():
            return candidate, candidate
    raise FileNotFoundError("Cannot locate repository root with Raw Data and forecasting notebooks")

REPO_ROOT, PACKAGE_ROOT = find_package_paths()
RAW_DIR = REPO_ROOT / "Raw Data"
NB_DIR = PACKAGE_ROOT / "notebooks"
ARTIFACT_DIR = PACKAGE_ROOT / "artifacts"
REPORT_DIR = PACKAGE_ROOT / "reports"
OUTPUT_DIR = PACKAGE_ROOT / "outputs"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("PACKAGE_ROOT =", PACKAGE_ROOT)
print("RAW_DIR =", RAW_DIR)


## Pipeline Steps

In [ ]:
pipeline_steps = pd.DataFrame([
    {"step": 1, "name": "EDA Preprocessed", "description": "Audit raw data, reconcile targets, build daily summaries."},
    {"step": 2, "name": "Feature Engineering", "description": "Calendar, Tet, Black Friday, promo, raw seasonal profiles, target lag/rolling features."},
    {"step": 3, "name": "Base Models", "description": "Seasonal naive, raw-derived recovery seasonal baseline, Ridge."},
    {"step": 4, "name": "Time Series Data Preparing", "description": "Rolling-origin folds with future-row feature sanitization."},
    {"step": 5, "name": "Advanced Models", "description": "Ridge, LightGBM, XGBoost, CatBoost where installed."},
    {"step": 6, "name": "Ensemble + Calibration", "description": "OOF-selected weights plus raw-derived recovery anchor."},
    {"step": 7, "name": "Prediction", "description": "Validate sample Date order, non-negative outputs, export only when requested."},
])
pipeline_steps.to_csv(REPORT_DIR / "00_pipeline_steps.csv", index=False)
pipeline_steps


## Raw Data Manifest

In [ ]:
required_csv = [
    "customers.csv", "geography.csv", "inventory.csv", "orders.csv", "order_items.csv",
    "payments.csv", "products.csv", "promotions.csv", "returns.csv", "reviews.csv",
    "sales.csv", "sample_submission.csv", "shipments.csv", "web_traffic.csv",
]
manifest = []
for name in required_csv:
    path = RAW_DIR / name
    assert path.exists(), f"Missing {name}"
    head = pd.read_csv(path, nrows=5)
    rows = sum(1 for _ in path.open("rb")) - 1
    manifest.append({"file": name, "rows": rows, "columns": "|".join(head.columns)})
manifest = pd.DataFrame(manifest)
manifest.to_csv(REPORT_DIR / "00_raw_data_manifest.csv", index=False)
manifest


## Sample Submission Audit

In [ ]:
sample_header = pd.read_csv(RAW_DIR / "sample_submission.csv", nrows=0)
sample_dates = pd.read_csv(RAW_DIR / "sample_submission.csv", usecols=["Date"], parse_dates=["Date"])
audit = {
    "sample_columns_available": sample_header.columns.tolist(),
    "loaded_columns": ["Date"],
    "row_count": int(len(sample_dates)),
    "start_date": str(sample_dates["Date"].min().date()),
    "end_date": str(sample_dates["Date"].max().date()),
}
assert audit["loaded_columns"] == ["Date"]
assert audit["row_count"] == 548
(REPORT_DIR / "00_sample_submission_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
audit


## Target Reconciliation

In [ ]:
sales = pd.read_csv(RAW_DIR / "sales.csv", parse_dates=["Date"])
orders = pd.read_csv(RAW_DIR / "orders.csv", usecols=["order_id", "order_date"], parse_dates=["order_date"])
items = pd.read_csv(RAW_DIR / "order_items.csv", usecols=["order_id", "product_id", "quantity", "unit_price"])
products = pd.read_csv(RAW_DIR / "products.csv", usecols=["product_id", "cogs"])

calc = (
    items.merge(products, on="product_id", how="left")
         .merge(orders, on="order_id", how="left")
)
calc["Date"] = calc["order_date"].dt.normalize()
calc["Revenue_calc"] = calc["quantity"] * calc["unit_price"]
calc["COGS_calc"] = calc["quantity"] * calc["cogs"]
daily = calc.groupby("Date", as_index=False)[["Revenue_calc", "COGS_calc"]].sum()
check = sales.merge(daily, on="Date", how="left")
result = {
    "max_abs_revenue_diff": float((check["Revenue"] - check["Revenue_calc"]).abs().max()),
    "max_abs_cogs_diff": float((check["COGS"] - check["COGS_calc"]).abs().max()),
    "sales_rows": int(len(sales)),
}
(REPORT_DIR / "00_target_reconciliation.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
result


## Compliance Notes

- Runtime inputs come from package-local `Raw Data/*.csv`.
- `sample_submission.csv` is used only for `Date`, row order, and schema validation.
- No old submission, sample target value, leaderboard-derived scale, or helper artifact is used.
